<a href="https://colab.research.google.com/github/SahanRachintha/Spaceship-Titanic/blob/main/Spaceship_Titanic_Trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
train_df = pd.read_csv('train (1).csv')
test_df = pd.read_csv('test (1).csv')

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [ ]:
num_cols=['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
cat_cols=['HomePlanet','Destination']
bool_cols=['CryoSleep','VIP']

In [ ]:
for col in bool_cols:
  train_df[col]=train_df[col].astype(str)
  test_df[col]=test_df[col].astype(str)

In [ ]:
for col in bool_cols:
  train_df[col]=train_df[col].fillna('Unknown')
  test_df[col]=test_df[col].fillna('Unknown')

In [ ]:
for col in num_cols:
  train_df[col]=train_df[col].fillna(train_df[col].mean())
  test_df[col]=test_df[col].fillna(test_df[col].mean())

In [ ]:
for col in cat_cols:
  train_df[col]=train_df[col].fillna('Unknown')
  test_df[col]=test_df[col].fillna('Unknown')

In [ ]:
train_df['Total_spending']=train_df['RoomService']+train_df['FoodCourt']+train_df['ShoppingMall']+train_df['Spa']+train_df['VRDeck']
test_df['Total_spending']=test_df['RoomService']+test_df['FoodCourt']+test_df['ShoppingMall']+test_df['Spa']+test_df['VRDeck']

In [ ]:
train_df['Cabin'].str.count('/').value_counts()

,count
Cabin,
2.0,8494


In [ ]:
# Temporary split to inspect components
temp = train_df['Cabin'].str.split('/', expand=True)
temp.columns = ['part1', 'part2', 'part3']

print("Unique values in part1 (likely Deck):")
print(temp['part1'].value_counts().head())
# Output: B, F, A, G, E... → Single letters = decks

print("\nSample values in part2 (likely Number):")
print(temp['part2'].head())
# Output: 0, 1234, 456, 734... → Numeric = cabin numbers

print("\nUnique values in part3 (likely Side):")
print(temp['part3'].value_counts())
# Output: S, P, nan → Only 2 values = Port/Starboard

Unique values in part1 (likely Deck):
part1
F    2794
G    2559
E     876
B     779
C     747
Name: count, dtype: int64

Sample values in part2 (likely Number):
0    0
1    0
2    0
3    0
4    1
Name: part2, dtype: object

Unique values in part3 (likely Side):
part3
S    4288
P    4206
Name: count, dtype: int64


In [ ]:
# Step 1: Create missingness indicators BEFORE splitting
train_df['Cabin_missing'] = train_df['Cabin'].isna().astype(int)
test_df['Cabin_missing'] = test_df['Cabin'].isna().astype(int)

# Step 2: Fill with DISTINCT placeholder unlikely to exist in real data
train_df['Cabin'] = train_df['Cabin'].fillna('X/9999/X')  # NOT 'Unknown/0/U'
test_df['Cabin'] = test_df['Cabin'].fillna('X/9999/X')

# Step 3: Split normally
train_df[['Deck', 'Num', 'Side']] = train_df['Cabin'].str.split('/', expand=True)
test_df[['Deck', 'Num', 'Side']] = test_df['Cabin'].str.split('/', expand=True)

# Step 4: Convert Num to numeric (9999 will stand out as placeholder)
train_df['Num'] = pd.to_numeric(train_df['Num'], errors='coerce')
test_df['Num'] = pd.to_numeric(test_df['Num'], errors='coerce')

train_df.drop('Cabin', axis=1, inplace=True)
test_df.drop('Cabin', axis=1, inplace=True)

In [ ]:
label_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']


In [ ]:
for col in label_cols:
  le=LabelEncoder()
  train_df[col]=le.fit_transform(train_df[col])
  test_df[col]=le.transform(test_df[col])

In [ ]:
x=train_df.drop(['Transported','PassengerId','Name'],axis=1)
y=train_df['Transported'].astype(int)


In [ ]:
x_train,x_val,y_train,y_val=train_test_split(x,y,test_size=0.2,random_state=42)


In [ ]:
rf=RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    random_state=42
)

In [ ]:
rf.fit(x_train,y_train)

RandomForestClassifier(max_depth=10, n_estimators=500, random_state=42)

In [ ]:
y_pred=rf.predict(x_val)
accuracy=accuracy_score(y_val,y_pred)
print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.78953421506613


In [ ]:
x_test = test_df.drop(['PassengerId','Name'], axis=1)
test_preds = rf.predict(x_test)

submission = pd.DataFrame({
    "PassengerId": test_df['PassengerId'],
    "Transported": test_preds.astype(bool)
})

submission.to_csv("submission.csv", index=False)